# Week 9 — Bayesian Optimization

Generate optimized recommendations for Week 9 using utility modules.

## Setup

In [ ]:
import numpy as np
import warnings
import sys
import importlib
sys.path.append('..')  # Add parent directory to path

# Import utility modules (reload to pick up any code changes)
import utils.bayesian_optimization
importlib.reload(utils.bayesian_optimization)
from utils.bayesian_optimization import propose_next_point, fit_gp, get_strategy

from utils.data_utils import (
    load_week_data,
    save_week_data,
    combine_with_week_results, 
    print_data_summary
)

## 1. Load Week 8 Data

Load the combined data from previous weeks

In [ ]:
# Load Week 8 clean data
inputs, outputs = load_week_data("../week 8/week8_clean_data.npz")
print_data_summary(inputs, outputs, "Week 8 Data")

## 2. Add Week 8 Results

In [ ]:
# Week 8 submitted points (actual values submitted)
week8_inputs = {
    1: np.array([0.423000, 0.415000]),
    2: np.array([0.741438, 0.079824]),
    3: np.array([0.348863, 0.672420, 0.439172]),
    4: np.array([0.413690, 0.367443, 0.355391, 0.413441]),
    5: np.array([1.000000, 0.000000, 1.000000, 1.000000]),
    6: np.array([0.755469, 0.280580, 0.644099, 0.672228, 0.162862]),
    7: np.array([0.000000, 0.314263, 0.707844, 0.246481, 0.405689, 0.758028]),
    8: np.array([0.107475, 0.120302, 0.020000, 0.212071, 1.000000, 0.099881, 0.180000, 0.998620])
}

# Week 8 outputs (received from black box)
week8_outputs = {
    1: 0.8992164200136958,
    2: 0.1272906879012989,
    3: -0.007200742877386392,
    4: 0.7152226612288932,
    5: 4440.5225,
    6: -0.5300701518003554,
    7: 1.8664977190350815,
    8: 9.762068463904
}

# Combine with Week 8 results
inputs, outputs = combine_with_week_results(inputs, outputs, week8_inputs, week8_outputs)
print_data_summary(inputs, outputs, "After Week 8 Results")

In [ ]:
# Save combined data for Week 9
save_week_data(inputs, outputs, "week9_clean_data.npz")

## 3. Week 8 Results Analysis

Evaluate which strategies worked and which failed to inform Week 9 approach.

In [ ]:
# Week 8 results analysis
print("=" * 70)
print("WEEK 8 RESULTS ANALYSIS")
print("=" * 70)

# Best values before Week 8 query
best_before_w8 = {}
for fid in range(1, 9):
    best_before_w8[fid] = np.max(outputs[fid][:-1])

print(f"\n{'F':>2} {'Dims':>4} {'Best Before W8':>14} {'W8 Query':>14} {'New Best':>14} {'Status'}")
print("-" * 70)

improved = 0
for fid in range(1, 9):
    dim = inputs[fid].shape[1]
    prev_best = best_before_w8[fid]
    w8_val = week8_outputs[fid]
    new_best = np.max(outputs[fid])
    
    if w8_val >= prev_best:
        status = "NEW BEST"
        improved += 1
    else:
        status = f"miss (best still {prev_best:.4f})"
    
    print(f"{fid:>2} {dim:>3}D {prev_best:>14.4f} {w8_val:>14.4f} {new_best:>14.4f}   {status}")

print(f"\nWeek 8 hit rate: {improved}/8 functions improved")
print("=" * 70)

In [ ]:
# Detailed Week 8 strategy evaluation
print("=" * 70)
print("WEEK 8 STRATEGY EVALUATION — What worked, what didn't")
print("=" * 70)

strategies_w8 = {
    1: ("Manual dim1+0.005", "NEW BEST +0.021 (0.879→0.899). 6th consecutive improvement! Alternating dims confirmed."),
    2: ("UCB-derived [0.741, 0.080]", "MISS 0.127 — worst F2 result ever. Stayed in the same dim1>0.67 cluster. Still haven't explored low dim1."),
    3: ("Manual dim1+0.001", "MISS -0.0072 vs best -0.0056. Better than W7's -0.0077. Bracketed: optimum between dim1=0.347-0.349."),
    4: ("Manual dim3-0.005", "MISS 0.715 vs best 0.724. dim3=0.355 overshot — trend broke. Sweet spot is 0.360-0.365."),
    5: ("EXPLORE [1,0,1,1]", "INFO ONLY 4441 (best remains 8662). Learned dim2 contributes ~4200 points (8662-4441)."),
    6: ("Manual dim2+0.005", "MISS -0.530 vs best -0.521. dim2=0.281 slightly worse. Optimum near dim2=0.276."),
    7: ("Manual dim2-0.003", "NEW BEST +0.005 (1.862→1.866). 3rd consecutive win from dim2 decrease. Tightening works."),
    8: ("Manual dim4+0.005 from W4", "MISS 9.7621 vs best 9.7627. Near-miss by 0.0006. dim4=0.212 slightly past optimum.")
}

for fid in range(1, 9):
    strategy, evaluation = strategies_w8[fid]
    print(f"\nF{fid}: {strategy}")
    print(f"  → {evaluation}")

print(f"\n{'=' * 70}")
print("SUMMARY: 2/8 new bests (F1, F7) — both from manual single-dim nudges")
print("Key insights: F4 dim3 trend broke at 0.355, F6 optimum near 0.276, F8 converged near dim4=0.207")
print("=" * 70)

In [ ]:
# Full history tracker — best score per function per week
print("=" * 70)
print("CUMULATIVE BEST SCORES ACROSS ALL WEEKS")
print("=" * 70)

print(f"\n{'F':>2} {'W1':>9} {'W2':>9} {'W3':>9} {'W4':>9} {'W5':>9} {'W6':>9} {'W7':>9} {'W8':>9}")
print("-" * 85)

week_results = {
    1: [0.0979, 3.1e-39, 0.3255, 0.4147, 0.6114, 0.7062, 0.8787, 0.8992],
    2: [0.5567, 0.6138, 0.0480, 0.6050, 0.5471, 0.5725, 0.5824, 0.1273],
    3: [-0.0593, -0.0499, -0.1750, -0.0056, -0.0701, -0.0079, -0.0077, -0.0072],
    4: [-4.4163, 0.3523, 0.4226, 0.6723, 0.7101, 0.4657, 0.7243, 0.7152],
    5: [1231.61, 1688.07, 7599.50, 8662.48, 8290.38, 8643.15, 1616.64, 4440.52],
    6: [-0.5920, -0.5210, -1.0560, -0.5902, -0.9899, -0.5207, -0.6062, -0.5301],
    7: [1.3646, 1.7845, 1.3720, 1.7718, 1.4720, 1.8536, 1.8617, 1.8665],
    8: [9.5863, 9.6493, 9.6972, 9.7627, 9.7274, 9.7402, 9.7614, 9.7621]
}

for fid in range(1, 9):
    running_best = []
    current_best = float('-inf')
    for val in week_results[fid]:
        current_best = max(current_best, val)
        running_best.append(current_best)
    
    vals = ' '.join(f'{v:>9.4f}' for v in running_best)
    print(f"{fid:>2} {vals}")

print(f"\n{'F':>2} {'Overall Best':>14} {'Best Week':>10}")
print("-" * 30)
for fid in range(1, 9):
    best_val = max(week_results[fid])
    best_week = week_results[fid].index(best_val) + 1
    print(f"{fid:>2} {best_val:>14.4f} {'W' + str(best_week):>10}")
print("=" * 70)

## 4. Sensitivity Analysis

Updated sensitivity analysis with Week 8 data (18 points per function) to inform Week 9 strategies.

In [ ]:
from utils.sensitivity import sensitivity_analysis

for func_id in range(1, 9):
    sensitivity_analysis(func_id, inputs[func_id], outputs[func_id])

## 5. Week 9 Strategy Design — NEW APPROACHES

After analysing all 8 weeks of strategies, results, and GP diagnostics, Week 9 breaks from the conservative micro-nudge playbook. Key insight: **we've been repeating the same dimension on each function for weeks while ignoring equally sensitive untested dimensions.**

### Week 8 lessons learned:
- **F1**: dim2 nudges give 3-8x bigger gains than dim1 (W7: +0.173 vs W8: +0.021). Combine both.
- **F2**: dim2 is IRRELEVANT (ls=1.015). Only dim1 matters. All 8 queries had dim1 > 0.67 — the second peak must be at different dim1.
- **F3**: dim1 is bracketed at 0.348. dim2 was tried (W6, failed). **dim3 has NEVER been touched** from best — ls=0.004, same as dim1.
- **F4**: Smooth landscape (all ls~1.5), 38 data points. GP should be reliable now. We've ONLY tweaked dim3 for 4 weeks — 3 other equally important dims are untouched.
- **F6**: Stuck at -0.52 tweaking dim2 for 4 weeks. dims 4&5 are ultra-sensitive — that's where untapped gains live.
- **F7**: dim5 (ls=0.26) has nearly identical sensitivity to dim2 (ls=0.25). Never tried from best point.
- **F8**: dim3 is the most sensitive active dim (ls=1.87). Untouched from best while we fixated on dim4.

### Strategy per function:

| F | Old approach | **New approach** | Rationale |
|---|-------------|-----------------|-----------|
| F1 | Alternate single dim +0.005 | **Both dims +0.005 → [0.428, 0.420]** | Both directions proven positive over 6 weeks. Combine for bigger jump |
| F2 | Micro-nudges near dim1=0.7 | **[0.20, 0.50] — deep low-dim1 exploration** | dim2 irrelevant (ls=1.0). Peer's 0.829 peak at unknown dim1. Maximise distance from cluster |
| F3 | dim1/dim2 micro-nudges | **dim3-0.002 → [0.348, 0.672, 0.437]** | Only dim never tested from best. ls=0.004, so 0.002 = half length scale. Controlled risk |
| F4 | Manual dim3 only | **GP/EI tight bounds (±0.03)** | Smooth landscape, 38 pts. Let optimizer search ALL 4 dims, not just dim3 |
| F5 | Corner exploration | **[0, 1, 1, 1]** — complete factorial | Isolates dim1's independent contribution |
| F6 | dim2 ±0.005 (stuck 4 weeks) | **dim5+0.0001** — micro-nudge ultra-sensitive dim | dim2 exhausted. dim5 ls=0.0003, so 0.0001 = 1/3 length scale. High ceiling |
| F7 | dim2 decrease (3 weeks) | **dim5-0.005 → [0.0, 0.314, 0.708, 0.246, 0.401, 0.758]** | Same sensitivity as dim2 (ls=0.26 vs 0.25). Fresh dimension, untapped potential |
| F8 | dim4 triangulation (3 weeks) | **dim3+0.010 → [0.107, 0.120, 0.030, 0.207, ...]** | Most sensitive active dim (ls=1.87). Untouched from best. dim4 is converged |

In [ ]:
import warnings
import importlib
import utils.bayesian_optimization
importlib.reload(utils.bayesian_optimization)
from utils.bayesian_optimization import fit_gp, propose_next_point

week9_recommendations = {}

def get_best_point(fid):
    """Get the best observed point for a function"""
    best_idx = np.argmax(outputs[fid])
    return inputs[fid][best_idx].copy()

# ============================================================
# F1: COMBO MOVE — both dims +0.005 → [0.428, 0.420]
# NEW APPROACH: Stop alternating one dim at a time.
# Both dims individually proven positive over 6 weeks.
# dim2 gains: +0.196 (W5), +0.173 (W7). dim1 gains: +0.095 (W6), +0.021 (W8).
# Combining should give bigger jump than either alone.
# ============================================================
best1 = get_best_point(1)  # [0.423, 0.415] → 0.8992
week9_recommendations[1] = np.array([best1[0] + 0.005, best1[1] + 0.005])

# ============================================================
# F2: DEEP EXPLORATION — [0.20, 0.50] — far from all known points
# NEW APPROACH: Go deeper into unexplored territory.
# dim2 is IRRELEVANT (ls=1.015) — only dim1 matters.
# All 8 queries had dim1 in [0.679-0.813]. Peer found 0.829 peak.
# [0.20, 0.50] maximises distance from our cluster.
# ============================================================
week9_recommendations[2] = np.array([0.200000, 0.500000])

# ============================================================
# F3: UNTESTED DIMENSION — dim3-0.002 → [0.348, 0.672, 0.437]
# NEW APPROACH: Switch to the only dim never touched from best.
# dim1 bracketed (0.347/0.348/0.349 all tested). dim2 tried (W6 failed).
# dim3 ls=0.004 (same as dim1) — equally sensitive, never tested.
# 0.002 step = half the length scale. Controlled exploration.
# ============================================================
best3 = get_best_point(3)  # [0.347863, 0.672420, 0.439172] → -0.0056
week9_recommendations[3] = np.array([best3[0], best3[1], best3[2] - 0.002])

# ============================================================
# F4: GP/EI WITH TIGHT BOUNDS — let optimizer search all 4 dims
# NEW APPROACH: Stop manual dim3-only tweaks.
# Smooth landscape (all ls ~1.5-1.7), 38 data points — GP is reliable.
# Tight bounds (±0.03) prevent wild proposals while exploring all dims.
# We've ignored dims 1, 2, 4 for 4 weeks — the optimizer won't.
# ============================================================
best4 = get_best_point(4)  # [0.413690, 0.367443, 0.360391, 0.413441] → 0.7243
bounds_f4 = np.array([
    [best4[0] - 0.03, best4[0] + 0.03],
    [best4[1] - 0.03, best4[1] + 0.03],
    [best4[2] - 0.03, best4[2] + 0.03],
    [best4[3] - 0.03, best4[3] + 0.03]
])
bounds_f4 = np.clip(bounds_f4, 0.0, 1.0)

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    pt4, _ = propose_next_point(
        inputs[4], outputs[4], bounds_f4,
        acq_func='EI', xi=0.01, n_restarts=50
    )
week9_recommendations[4] = pt4

# ============================================================
# F5: FACTORIAL EXPLORATION — [0, 1, 1, 1]
# Complete the corner design:
# [1,1,1,1]=8662, [1,0,1,1]=4441, [0,0,1,1]=1617.
# [0,1,1,1] isolates dim1: effect = 8662 - [0,1,1,1]
# ============================================================
week9_recommendations[5] = np.array([0.000000, 1.000000, 1.000000, 1.000000])

# ============================================================
# F6: ULTRA-SENSITIVE DIM — dim5+0.0001
# NEW APPROACH: Break out of dim2 stagnation (stuck at -0.52 for 4 weeks).
# dim5 has ls=0.0003 — the most sensitive dimension in the entire project.
# 0.0001 = 1/3 of a length scale. Controlled but potentially game-changing.
# Unlike W5's disaster (moved ALL dims), this is a single micro-nudge.
# ============================================================
best6 = get_best_point(6)  # [0.755469, 0.275580, 0.644099, 0.672228, 0.162862] → -0.5207
week9_recommendations[6] = np.array([
    best6[0],            # dim1 locked
    best6[1],            # dim2 locked (exhausted)
    best6[2],            # dim3 locked
    best6[3],            # dim4 locked
    best6[4] + 0.0001    # dim5 +0.0001 (1/3 of length scale 0.0003)
])

# ============================================================
# F7: FRESH DIMENSION — dim5-0.005
# NEW APPROACH: Stop riding dim2 (3 weeks same strategy).
# dim5 has ls=0.26 — nearly identical sensitivity to dim2 (ls=0.25).
# dim5=0.406 in best. Try 0.401. Untapped potential.
# If this works, we have TWO productive dimensions to alternate.
# ============================================================
best7 = get_best_point(7)  # [0.0, 0.314263, 0.707844, 0.246481, 0.405689, 0.758028] → 1.8665
week9_recommendations[7] = np.array([
    best7[0],            # dim1 locked at 0.0
    best7[1],            # dim2 locked (preserve gains)
    best7[2],            # dim3 locked
    best7[3],            # dim4 locked
    best7[4] - 0.005,    # dim5 -0.005 (0.406→0.401) — FRESH DIM
    best7[5]             # dim6 locked
])

# ============================================================
# F8: NEW DIMENSION — dim3+0.010
# NEW APPROACH: dim4 is converged (0.207 vs 0.212, difference 0.0006).
# dim3 has the SMALLEST length scale of active dims (ls=1.87).
# dim3=0.020 in best point — very low. 0.010 step = 0.005 length scales.
# This is where the next gain is most likely to come from.
# ============================================================
best8 = get_best_point(8)  # W4 best: [0.107, 0.120, 0.020, 0.207, 1.0, 0.100, 0.180, 0.999]
week9_recommendations[8] = np.array([
    best8[0],            # dim1 locked
    best8[1],            # dim2 locked
    best8[2] + 0.010,    # dim3 +0.010 (0.020→0.030) — FRESH DIM
    best8[3],            # dim4 locked at 0.207
    best8[4],            # dim5 locked at 1.0
    best8[5],            # dim6 locked (irrelevant)
    best8[6],            # dim7 locked
    best8[7]             # dim8 locked (irrelevant)
])

# ============================================================
# Sanity check all recommendations
# ============================================================
print("Week 9 Recommendations — NEW APPROACHES")
print("=" * 80)
for fid in range(1, 9):
    X, y = inputs[fid], outputs[fid]
    rec = week9_recommendations[fid]
    
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        gp = fit_gp(X, y)
    pred, pred_std = gp.predict(rec.reshape(1, -1), return_std=True)
    dists = np.linalg.norm(X - rec, axis=1)
    min_dist = np.min(dists)
    best = np.max(y)
    
    strategy = {
        1: "COMBO both dims +0.005 (proven directions combined)",
        2: "DEEP EXPLORE [0.20, 0.50] — max distance from cluster",
        3: "FRESH DIM dim3-0.002 (only untested dim from best)",
        4: "GP/EI tight bounds ±0.03 (all 4 dims, not just dim3)",
        5: "FACTORIAL [0,1,1,1] — isolate dim1 effect",
        6: "ULTRA-SENSITIVE dim5+0.0001 (break dim2 stagnation)",
        7: "FRESH DIM dim5-0.005 (same sensitivity as dim2, untapped)",
        8: "FRESH DIM dim3+0.010 (most sensitive active dim, untouched)"
    }
    
    print(f"F{fid} ({X.shape[1]}D)  best={best:.4f}  pred={pred[0]:.4f}±{pred_std[0]:.4f}  dist={min_dist:.4f}  {strategy[fid]}")
    print(f"  point: {rec}")
print("=" * 80)

## 6. Submission Format

In [ ]:
# Submission format
print("=" * 70)
print("WEEK 9 SUBMISSION")
print("=" * 70)

for fid in range(1, 9):
    pt = week9_recommendations[fid]
    formatted = '-'.join(f'{x:.6f}' for x in pt)
    print(f"Function {fid}:\t{formatted}")